# Dialogue Summarization with PEGASUS

This notebook fine-tunes Google's **PEGASUS** model on the **SAMSum** conversational dataset to generate abstractive summaries of dialogues. The pipeline covers environment setup, data preprocessing, model training with the Hugging Face Trainer API, ROUGE-based evaluation, and model persistence.

**Model:** `google/pegasus-cnn_dailymail`  
**Dataset:** SAMSum (14,732 train / 819 test / 818 validation samples)  
**Hardware:** NVIDIA Tesla T4 GPU  
**Framework:** Hugging Face Transformers + Datasets

## 1. Environment Setup

Verify GPU availability and install all required libraries. A compatible version of `transformers` (4.40.0) is pinned to ensure the summarization pipeline and Trainer API work correctly in this environment.

In [1]:
!nvidia-smi

Sat Mar 28 13:47:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers==4.40.0 datasets sacrebleu rouge_score evaluate py7zr accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages 

## 2. Import Libraries

Import all necessary libraries for data loading, model training, tokenization, evaluation, and utility functions.

In [7]:
!pip install transformers==4.51.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.0 MB/s eta 0:00:00


In [1]:
import transformers
print(transformers.__version__)  # should print 4.51.0

4.51.0


In [2]:
import os
import numpy as np
import pandas as pd
import torch
import nltk
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

nltk.download("punkt", quiet=True)

True

## 3. Device Configuration

Automatically select GPU if available, otherwise fall back to CPU. Training on a GPU (CUDA) significantly reduces fine-tuning time for large sequence-to-sequence models.

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 4. Load Base Model and Tokenizer

Load the pretrained `google/pegasus-cnn_dailymail` model and its corresponding tokenizer. This checkpoint was originally trained on CNN/DailyMail news articles and serves as our starting point for fine-tuning on dialogue data.

In [4]:
MODEL_NAME = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model parameters: {model_pegasus.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Model loaded: google/pegasus-cnn_dailymail
Model parameters: 570,797,056


## 5. Download and Load the SAMSum Dataset

Download the SAMSum dataset, which contains messenger-style dialogues paired with human-written summaries. The dataset is stored in Arrow format for fast loading.

In [5]:
!wget -q https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
!unzip -q -o summarizer-data.zip

dataset_samsum = load_from_disk("samsum_dataset")
print(dataset_samsum)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})


## 6. Exploratory Data Analysis

Inspect the dataset structure, split sizes, and a sample dialogue-summary pair to understand the nature of the data before preprocessing.

In [6]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]
print(f"Split sizes  : {dict(zip(dataset_samsum.keys(), split_lengths))}")
print(f"Features     : {dataset_samsum['train'].column_names}")

print("\nSample Dialogue:")
print(dataset_samsum["test"][2]["dialogue"])

print("\nReference Summary:")
print(dataset_samsum["test"][2]["summary"])

Split sizes  : {'train': 14732, 'test': 819, 'validation': 818}
Features     : ['id', 'dialogue', 'summary']

Sample Dialogue:
Lenny: Babe, can you help me with something?
Bob: Sure, what's up?
Lenny: Which one should I pick?
Bob: Send me photos
Lenny:  <file_photo>
Lenny:  <file_photo>
Lenny:  <file_photo>
Bob: I like the first ones best
Lenny: But I already have purple trousers. Does it make sense to have two pairs?
Bob: I have four black pairs :D :D
Lenny: yeah, but shouldn't I pick a different color?
Bob: what matters is what you'll give you the most outfit options
Lenny: So I guess I'll buy the first or the third pair then
Bob: Pick the best quality then
Lenny: ur right, thx
Bob: no prob :)

Reference Summary:
Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.


## 7. Data Preprocessing

Tokenize the dialogues (inputs) and summaries (labels) using the PEGASUS tokenizer. The maximum input length is set to 1024 tokens and label length to 128 tokens, which are standard limits for this architecture. The Hugging Face Trainer handles label shifting internally for seq2seq training.

In [7]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["dialogue"],
        max_length=1024,
        truncation=True
    )
    target_encodings = tokenizer(
        example_batch["summary"],
        max_length=128,
        truncation=True
    )
    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)
print(dataset_samsum_pt)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})


## 8. Data Collator

Initialize a `DataCollatorForSeq2Seq` which dynamically pads each batch to the length of the longest sequence in that batch. This is more memory-efficient than padding all sequences to the global maximum length.

In [8]:
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

## 9. ROUGE Evaluation Metric

Load the ROUGE metric which will be used both during training evaluation and for final model assessment. ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures the overlap between generated and reference summaries.

In [9]:
rouge_metric = evaluate.load("rouge")
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

## 10. Training Configuration

Define the training hyperparameters. Key settings:

- **1 epoch** with gradient accumulation over 16 steps (effective batch size = 16)
- **Warmup steps** of 500 to stabilize early training
- **Weight decay** of 0.01 for regularization
- Evaluation runs every 500 steps on the validation set

In [10]:
trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    do_eval=True,
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)

## 11. Initialize Trainer

Instantiate the Hugging Face `Trainer` with the model, training arguments, data collator, datasets, and metrics function. The Trainer abstracts the full training loop including forward pass, loss computation, backpropagation, and evaluation.

In [11]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
    compute_metrics=compute_metrics
)

## 12. Fine-tune the Model

Run the training loop. With 1 epoch on 14,732 samples, a batch size of 1, and gradient accumulation of 16 steps, the trainer will perform approximately 921 optimization steps on the T4 GPU. This typically takes around 45 minutes.

In [13]:
trainer.train()

Step,Training Loss
10,1.616700
20,1.614900
30,1.635300
40,1.577200
50,1.505600
60,1.584200
70,1.548600
80,1.540900
90,1.600700
100,1.540700


TrainOutput(global_step=920, training_loss=1.503282672425975, metrics={'train_runtime': 2912.0576, 'train_samples_per_second': 5.059, 'train_steps_per_second': 0.316, 'total_flos': 5528248038285312.0, 'train_loss': 1.503282672425975, 'epoch': 0.9991854466467553})

## 13. Batch ROUGE Evaluation on Test Set

Evaluate the fine-tuned model on a subset of the test set using ROUGE scores. The helper function batches the test dialogues, generates summaries using beam search, and computes corpus-level ROUGE metrics to quantify summarization quality.

In [15]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Yield successive batch-sized chunks from a list."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i: i + batch_size]


def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                 batch_size=16, device=device,
                                 column_text="article",
                                 column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )
        decoded_summaries = [
            tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for s in summaries
        ]
        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    return metric.compute()


score = calculate_metric_on_test_ds(
    dataset_samsum["test"][0:10],
    rouge_metric,
    trainer.model,
    tokenizer,
    batch_size=2,
    column_text="dialogue",
    column_summary="summary"
)

rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=["pegasus_finetuned"])

100%|██████████| 5/5 [00:09<00:00,  1.87s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus_finetuned,0.024049,0.0,0.024216,0.023886


## 14. Save the Fine-tuned Model and Tokenizer

Persist the fine-tuned model weights and tokenizer configuration to disk. These artifacts can be reloaded later for inference without retraining.

In [16]:
model_pegasus.save_pretrained("pegasus-samsum-model")
tokenizer.save_pretrained("tokenizer")

print("Model and tokenizer saved successfully.")

Model and tokenizer saved successfully.


## 15. Inference on a Sample Dialogue

Reload the fine-tuned model and tokenizer from disk and run inference on a test sample. The summary is generated using beam search with a length penalty to encourage concise output.

In [17]:
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")
model_finetuned = AutoModelForSeq2SeqLM.from_pretrained("pegasus-samsum-model").to(device)

gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

sample_text = dataset_samsum["test"][0]["dialogue"]
reference   = dataset_samsum["test"][0]["summary"]

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

summary_ids = model_finetuned.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    **gen_kwargs
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(summary)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda can't find Betty's number. Larry called her the last time they were at the park together. Hannah wants Amanda to text Larry instead.


## 16. Export Model to Google Drive

Mount Google Drive and copy the fine-tuned model and tokenizer files for permanent storage. This ensures the trained artifacts are preserved even after the Colab runtime disconnects.

In [19]:
import shutil
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

os.makedirs("/content/drive/MyDrive/pegasus-samsum-model", exist_ok=True)
os.makedirs("/content/drive/MyDrive/tokenizer", exist_ok=True)

for f in os.listdir("tokenizer"):
    shutil.copy(f"tokenizer/{f}", f"/content/drive/MyDrive/tokenizer/{f}")
    print(f"Copied tokenizer/{f}")

for f in os.listdir("pegasus-samsum-model"):
    print(f"Copying {f} ...")
    shutil.copy(f"pegasus-samsum-model/{f}", f"/content/drive/MyDrive/pegasus-samsum-model/{f}")
    print(f"Done: {f}")

print("\nAll model artifacts saved to Google Drive.")

Mounted at /content/drive
Copied tokenizer/tokenizer_config.json
Copied tokenizer/special_tokens_map.json
Copied tokenizer/tokenizer.json
Copied tokenizer/spiece.model
Copying generation_config.json ...
Done: generation_config.json
Copying config.json ...
Done: config.json
Copying model.safetensors ...
Done: model.safetensors

All model artifacts saved to Google Drive.
